# Char LM
이 노트북에서는 가장 기초적인 형태의 **decoder-only Transformer**를 구현합니다.

<img src="./img/charLM.png" width="800">

일반적인 서브워드 토큰 대신 각 글자(character)를 하나의 토큰으로 사용하는 character-level 방식을 사용하며, Transformer 기반 언어 모델의 기본 
구조와 학습 과정을 이해하는 것을 목표로 합니다.

## 1. text <-> token

### LLM의 학습 데이터

기본적인 **word-level tokenization**에서의 데이터 구성 방식은 다음과 같다.  
다만 실제 LLM에서는 보통 word-level보다 **subword tokenizer**를 사용한다.

예를 들어 다음 문장이 있다고 하자.

`나는 사과를 먹었다.`

Tokenizer가 이를 다음과 같이 변환한다고 가정하자.

| 토큰 |  | ID |
|---|:---:|---:|
| 나는 | → | 15 |
| 사과를 | → | 83 |
| 먹었다 | → | 233 |
| . | → | 3 |

그러면 문장은 다음과 같은 토큰 ID의 나열로 표현할 수 있다.


`tokens = [15, 83, 233, 3] `


LLM의 기본적인 학습 목표는 **이전 토큰들을 바탕으로 다음 토큰을 예측하는 것**이다.

따라서 입력과 정답을 한 칸씩 어긋나게 구성할 수 있다.


| 구분 | 토큰 |
|---|---|
| 입력 | [나는, 사과를, 먹었다] |
| 정답 |  [사과를, 먹었다, .] |

Causal Mask를 적용하면 하나의 시퀀스 안에서 다음과 같은 예측을 동시에 학습할 수 있다.

| 입력 |  | 정답 |
|---|:---:|---|
| 나는 | → | 사과를 |
| 나는 사과를 | → | 먹었다 |
| 나는 사과를 먹었다 | → | . |

즉, 토큰 시퀀스 `x`와 정답 시퀀스 `y`를 한 칸 차이 나게 구성하는 것만으로 **next-token prediction**을 위한 학습 데이터를 만들 수 있다.

한편 **character-level tokenization**에서는 단어 대신 문자 하나를 하나의 토큰으로 사용한다. 예를 들어 `사과`는 `사`, `과`라는 두 토큰으로 나뉠 수 있으며, 각각의 토큰은 모델 내부에서 하나의 정수형 **token ID**로 표현된다.

* input.txt는 한국어 위키백과의 인공지능 관련 문서에서 추출한 텍스트 3만자로로 구성되어 있습니다.

In [102]:
with open("input.txt", "r", encoding = "utf-8") as f:
    text = f.read()

print(text[:100])

인공지능(人工智能, 영어: artificial intelligence, AI)은 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 컴퓨터 과학의 세부분야 중 하나이다. 


### 전처리

위키백과의 위키문법을 제거하는 전처리를 진행한다.

In [103]:
import re

def preprocess(text):
    # 위키 문법 제거
    text = re.sub(r'=+', '', text)

    # HTML 공백
    text = text.replace('&#x20;', ' ')

    # 공백, 탭, 줄바꿈 ->  공백 1개
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

text = preprocess(text)

print(text[:100])
print("문자 수:", len(text))


인공지능(人工智能, 영어: artificial intelligence, AI)은 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 컴퓨터 과학의 세부분야 중 하나이다. 
문자 수: 29375


### 토크나이저 만들기

#### vocabulary 만들기

In [104]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

print(chars)
print(vocab_size)

[' ', '!', '"', '%', "'", '(', ')', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '^', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '~', '·', '‘', '’', '“', '”', '《', '》', '人', '工', '智', '能', '가', '각', '간', '갈', '감', '갑', '갔', '강', '갖', '같', '개', '객', '거', '걱', '건', '걸', '검', '것', '게', '겠', '겨', '격', '겪', '견', '결', '겼', '경', '계', '고', '곤', '곧', '곳', '공', '과', '관', '광', '괴', '교', '구', '국', '군', '굴', '궁', '권', '귀', '규', '균', '그', '극', '근', '글', '금', '급', '기', '긴', '길', '깊', '까', '깎', '깔', '깨', '께', '껴', '꼈', '꼬', '꼽', '꽉', '꾸', '꿈', '끈', '끊', '끌', '끔', '끝', '나', '낙', '난', '날', '남', '났', '낮', '내', '낸', '낼', '냈', '냉', '너', '넌', '널', '넓', '넘', '넛', '넣', '네', '넷', '년', '념', '녔', '노', '논', '놀', '높', '놓', '뇌', '누', '눈', '눌', '뉴', '느', '는', '늘',

#### tokenizer 만들기

##### 파이썬 문법

##### 1. enumerate
```python
enumerate(iterable)
```

`iterable`: 리스트, 문자열, 튜플처럼 반복 가능한 것

enumerate(chars)는 기본적으로 `(인덱스, 값)` 쌍을 만들어줌.

In [105]:
'''
for item in enumerate(chars):
    print(item)
'''

'''
print(list(enumerate(chars)))
'''

'''
for i,ch in enumerate(chars):
    print(i, ch)
'''


for i,ch in enumerate(chars):
    if i<=10: 
        print(i, ch)
    elif i == 11: print(".\n.\n.")
    elif i>=750: print(i, ch)

0  
1 !
2 "
3 %
4 '
5 (
6 )
7 +
8 ,
9 -
10 .
.
.
.
750 흔
751 흥
752 희
753 히
754 힌
755 힐
756 힘


enumerate는 **반복 가능한 객체**를 만든다.

In [106]:
e = enumerate(chars)

print(next(e))

print(next(e))

print(next(e))

(0, ' ')
(1, '!')
(2, '"')


반복 가능한 객체의 각 원소에 인덱스를 붙이고, 반복할 때마다 다음 (인덱스, 값) 튜플을 내놓음

- 튜플 : 수정 불가 리스트

##### 2. dictionary comprehension 

```python
{key: value for item in iterable}
```

`(key, value)` 쌍의 딕셔너리를 만든다.


In [107]:
test_list = ['안', '녕', '하', '세', '요']
test_dictionary = {i : j for i, j in enumerate(test_list)}

print(test_list)
print(test_dictionary)

['안', '녕', '하', '세', '요']
{0: '안', 1: '녕', 2: '하', 3: '세', 4: '요'}


In [108]:
# 문자 -> 숫자
stoi = {ch: i for i, ch in enumerate(chars)}

# 숫자 -> 문자
itos = {i: ch for i, ch in enumerate(chars)}

딕셔너리는 key를 입력하면 그 key에 해당하는 value가 나온다.

In [109]:
sample_stoi = {
    'a' : 65,
    'b' : 67,
    'c' : 68
}

print(sample_stoi['a'])


65


In [110]:
# 인코더
def encode(s):
    return [stoi[c] for c in s]

# 디코더
def decode(tokens):
    return "".join(itos[i] for i in tokens)

In [111]:
tokens = encode("인공지능")

print(tokens)
print(decode(tokens))

[534, 121, 575, 200]
인공지능


#### token (용어)

**토큰(token)**은 본질적으로 tokenizer가 텍스트를 나눈 하나의 단위이다.

char-level에서는
```
"안녕하세요" -> ["안", "녕", "하", "세", "요"] 
```

여기서 "안", "녕" 각각이 토큰이다.

그런데 컴퓨터는 문자열을 바로 연산하기 어려우니 각 토큰에 번호를 붙인다.

| 토큰 | Token ID |
|---|---:|
| `"안"` | 120 |
| `"녕"` | 241 |
| `"하"` | 701 |

이 `120`, `241`, `701`을 **token ID**라 부른다.

그 후 임베딩을 한 토큰을 표현하는 `[0.13, -0.72, ..., 0.08]` 이런 벡터를 token embedding 이라 한다.
하지만 실제 LLM 설명에서는 문맥상 편하게 셋 다 그냥 **토큰”*이라고 부르는 경우가 많다.

## 2. Dataset && DataLoader

In [112]:
%pip install numpy
%pip install torch


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### 전체 데이터를 Tensor로 변환

PyTorch에서는 토큰 ID를 보통 `torch.long`으로 사용한다.
이 프로젝트에서는 과해보일 수 있으나 임베딩 레이어로 인해 관습적으로 `torch.long`을 사용한다.

`torch.long`

= int64

= 64비트 정수

In [113]:
import torch

data = torch.tensor(encode(text),dtype=torch.long)

print(data.shape)
print(data[:20])


torch.Size([29375])
tensor([534, 121, 575, 200,   5,  85,  86,  87,  88,   8,   0, 492, 473,  22,
          0,  51,  68,  70,  59,  56])


### train / validation data 나누기

In [114]:
n = int(len(data) * 0.9)

train_data = data[:n]
val_data = data[n:]

### block size와 batch 사이즈

#### block size
LLM은 전체 텍스트 3만 자를 한 번에 넣지 않고 일정 길이만 잘라서 학습한다. 
여기서 block size는 Transformer가 한 번에 읽을 최대 길이를 정한다.

예를 들어
```
block_size = 8
```

block size만큼의 토큰을 가져와
```
t0 t1 t2 t3 t4 t5 t6 t7 t8
```

다음과 같이 입력과 정답 데이터를 구성한다.

|   | 토큰 |
|---|---|
| x | t0 t1 t2 t3 t4 t5 t6 t7 |
| y | t1 t2 t3 t4 t5 t6 t7 t8 |

이후 **causal masking**을 적용하여 각 위치가 미래 토큰을 보지 못하도록 한 상태에서 다음 토큰을 예측하도록 학습한다.

여기서 `block_size`가 나중에 말하는 **context length**다.
여기선 block size가 8이므로 llm이 다음 단어를 예측 할 때 이전 토큰 8개만 보고 다음 토큰을 예측하게 된다.


---

#### batch size

**chunk**

`chunk` 는 전체 토큰 데이터를 block size 길이만큼 잘라 만든 하나의 학습 샘플이다.
* **샘플** : 데이터셋을 구성하는 하나의 개별 데이터 예시 (ex. 이미지 모델 데이터 `이미지 1장 + 정답 라벨 1개`)


예를 들어 전체 토큰이

```
t0 t1 t2 t3 t4 t5 t6 t7 t8 t9 ...
```

이고 block size = 4라면 하나의 chunk는

```text
chunk 1: t0 t1 t2 t3
chunk 2: t1 t2 t3 t4
chunk 3: t2 t3 t4 t5
...
```

가 된다.

여기서 청크를 방식은 다양하게 존재한다. 대표적으로 두 가지가 있는데
**1. Sliding window 방식** 과 **2. Non-overlapping** 방식 이다.

1. Sliding window 방식
한 칸씩 밀면서 청크를 만든다.

```python
data = [0, 1, 2, 3, 4, 5, 6]
block_size = 4
```

```text
chunk 1 = [0, 1, 2, 3]
chunk 2 = [1, 2, 3, 4]
chunk 3 = [2, 3, 4, 5]
chunk 4 = [3, 4, 5, 6]
```

2. Non-overlapping 방식
청크 길이만큼 한 번에 이동한다.

```text
chunk 1 = [0, 1, 2, 3]
chunk 2 = [4, 5, 6, 7]
chunk 3 = [8, 9, 10, 11]
```

### Dataset과 DataLoader의 역할

#### Dataset
Dataset은 원본 데이터에서 **"학습 샘플 하나를 어떻게 만들지"** 정의한다.
즉, 이는 block size와 관련 있다. (입력, 출력 라벨쌍을 구성해야 하기 때문이다.)

```python
def __getitem__(self, idx):
    x = data[idx:idx + block_size]
    y = data[idx + 1:idx + block_size + 1]

    return x, y
```

`idx`에 따라
```text
sample 0
x = t0 t1 t2 t3
y = t1 t2 t3 t4

sample 1
x = t1 t2 t3 t4
y = t2 t3 t4 t5
```

---

#### DataLoader
DataLoader는 Dataset에서 만들어진 샘플들을 **"실제로 어떤 순서로 몇 개씩 가져올지"** 관리한다.
한 번에 학습에 사용할 샘플 수 또한 정하기에 block size와 관련 있다.

```python
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
)
```

주요 옵션은
| 옵션 | 역할 |
|---|---|
| `batch_size` | 한 번에 가져올 샘플 개수 |
| `shuffle` | epoch마다 샘플 순서를 섞을지 |
| `drop_last` | 마지막에 batch 크기보다 적게 남은 샘플을 버릴지 |
| `num_workers` | 데이터를 병렬로 불러오는 프로세스 수 |

##### **shuffle**

Dataset이 아래와 같이 구성되어 있고
```text
sample 0
sample 1
sample 2
sample 3
sample 4
sample 5
sample 6
sample 7
```


##### `shuffle=False` 이면?

```text
batch 1 = sample 0, 1, 2, 3
batch 2 = sample 4, 5, 6, 7
```

##### `shuffle=True` 이면?

순서 :5, 1, 7, 2, 0, 6, 3, 4


```text
batch 1 = sample 5, 1, 7, 2
batch 2 = sample 0, 6, 3, 4
```
---


### sequence 만들기
* sequence : 순서가 있는 토큰들의 묶음

In [115]:
block_size = 8

In [116]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

print(x)
print(y)

tensor([534, 121, 575, 200,   5,  85,  86,  87])
tensor([121, 575, 200,   5,  85,  86,  87,  88])


### batch 만들기

In [117]:
block_size = 8
batch_size = 4

##### 파이썬 문법

**삼항 조건문**

```
값1 if 조건 else 값2
```

조건이 참이면 값1 조건이 거짓이면 값2를 반환한다

---
**torch.randint()**

`torch.randint(low, high, size)`

| 인자 | 의미 |
|---|---|
| `low` | 시작값, 포함 |
| `high` | 끝값, 미포함 |
| `size` | 결과 텐서의 shape |

- low 생략
low 가 자동으로 0으로 간주된다.

```python
torch.randint(high, size)
```

```python
torch.randint(0, high, size)
```


**튜플**

size는 튜플로 지정한다.

`torch.randint`는 size에 지정한 shape의 Tensor를 반환한다.

1차원 Tensor를 만들고 싶을 때는 아래처럼 (4,)로 작성한다.
- python에서 원소가 1개인 튜플을 만들 때는 (n,)으로 표시한다. (n)은 그냥 정수 n이다.

```python
x = torch.randint(10, (4,))
```

2차원 이상은 다음처럼 작성하면 된다.
```python
torch.randint(10, (2, 3))
torch.randint(10, (2, 3, 4))
```

---

**torch.stack**

```
torch.stack(tensors, dim=0)
```

| 인자 | 의미 |
|---|---|
| `tensors` | 합칠 Tensor들의 리스트 또는 튜플 |
| `dim` | 새로 만들 차원의 위치 |

기본값은 dim=0

In [118]:
def get_batch(split):
    data = train_data if split == "train" else val_data

    ix = torch.randint(len(data)-block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x, y

In [119]:
xb, yb = get_batch("train")

print(xb)
print(yb)

print(xb.shape)
print(yb.shape)

tensor([[  0, 488, 127, 425, 483,   0, 539, 216],
        [198, 218,   8,   0, 533, 106, 528,   0],
        [  0, 734, 115, 483, 413, 220,   0, 437],
        [716, 206,  10,   0, 136, 198,   0, 394]])
tensor([[488, 127, 425, 483,   0, 539, 216,   0],
        [218,   8,   0, 533, 106, 528,   0, 331],
        [734, 115, 483, 413, 220,   0, 437, 437],
        [206,  10,   0, 136, 198,   0, 394, 522]])
torch.Size([4, 8])
torch.Size([4, 8])


### pytorch Dataset

**Dataset 선언 방식**

기본 형태

```python
from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, ...):
        ...

    def __len__(self):
        ...

    def __getitem__(self, idx):
        ...
```

Dataset의 역할은 idx를 주면 그 idx에 맞는 샘플을 반환하는 것이다.
위에서 설명했듯이 이는  1. Sliding window 와  2. Non-overlapping chunk로 구현 가능하다.
하지만 이번 노트북에서는 1번 방식을 사용한다.


#### 1.Sliding window

In [120]:
from torch.utils.data import Dataset


class TextDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[
            idx:idx + self.block_size
        ]

        y = self.data[
            idx + 1:idx + self.block_size + 1
        ]

        return x, y

#### 2. Non-overlapping chunk

```python
class ChunkDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return (len(self.data) - 1) // self.block_size

    def __getitem__(self, idx):
        start = idx * self.block_size

        x = self.data[start:start + self.block_size]
        y = self.data[start + 1:start + self.block_size + 1]

        return x, y
```

In [121]:
train_dataset = TextDataset(
    train_data,
    block_size = 8 
)

In [122]:
xb_0, yb_0 = train_dataset.__getitem__(0)

print(xb_0)
print(yb_0)

tensor([534, 121, 575, 200,   5,  85,  86,  87])
tensor([121, 575, 200,   5,  85,  86,  87,  88])


---

### pytorch DataLoader

**DataLoader 선언 방식**

이제 Dataset을 data, block_size를 인자로 받아 생성할 수 있다.
그러면 idx를 통해 각 샘플을 만들어낼 수 있는데, 

DataLoader는 이 샘플들을 **어떤 순서로, 몇 개씩 가져올지** 관리한다.

기본 형태는 
```python
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=4 ,
    shuffle=True
)
```

| 인자 | 역할 |
|---|---|
| `dataset` | 사용할 Dataset |
| `batch_size` | 한 번에 가져올 샘플 개수 |
| `shuffle` | 샘플 순서를 섞을지 여부 |


In [123]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
)

## 3. Embedding

#####  token && positional

### Token Embedding

지금까지의 입력은 (block_size, batch_size)이다.

```python
x.shape
# [B, T]
```

여기서 B는 batch size, T는 block size(sequence length)이다.

이제 각 token id에 해당하는 embedding vector를 가져와서, 해당 토큰의 표현으로 사용한다.

In [124]:
import torch
import torch.nn as nn

vocab_size = len(chars)
d_model = 256

token_embedding = nn.Embedding(vocab_size, d_model)

tok_emb = token_embedding(xb)
tok_emb.shape
# [B, T, d_model]

torch.Size([4, 8, 256])

### position embedding

self-attention은 기본적으로 입력 토큰의 순서 자체를 모른다.

그래서 이를 해결하기 위해 `position embedding`을 사용한다.
이는 위치값을 생각하기 때문에 기존 token embedding이 vocab_size를 이용했다면 이는 block size를 이용한다.

기존 transformer의 경우 **1. Sinusoidal Positional Encoding** 을 사용했지만 초기 gpt 계열은 **2. Learned Positional Embedding**를 사용한다.

---

#### 1. Sinusoidal Positional Encoding

<img src="./img/spe.png" width="300">

이는 Transformer 논문에서 사용한 방식이다.
각 자리 idx에 따른 sin, cos 패턴을 더해주는 방식이다.
이는 학습되지 않는다는 특징이 있다.

```python
pos = torch.arange(T).unsqueeze(1)

div = torch.exp(
    torch.arange(0, d_model, 2)
    * (-math.log(10000.0) / d_model)
)

pos_emb = torch.zeros(T, d_model)

pos_emb[:, 0::2] = torch.sin(pos * div)
pos_emb[:, 1::2] = torch.cos(pos * div)
```

---

#### 2.Learned Positional Embedding

위치도 토큰처럼 하나의 ID라고 생각하고
위치에 따른 학습되는 임베딩 벡터를 더해준다

```python
position_embedding = nn.Embedding(block_size, d_model)

pos = torch.arange(T, device=x.device)
pos_emb = position_embedding(pos)
```

##### 파이썬 문법

#### torch.arange

```
torch.arange(end)
```

0~end-1 까지를 원소로 가지는 텐서


In [125]:
tok_emb = token_embedding(x)   


position_embedding = nn.Embedding(block_size, d_model)

T = block_size

pos = torch.arange(T, device=x.device)
pos_emb = position_embedding(pos) 

In [126]:
class Embedding(nn.Module):
    def __init__(self, vocab_size, d_model, block_size):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)

    def forward(self, x):
        B, T = x.shape

        tok_emb = self.token_embedding(x)   # [B, T, C]

        pos = torch.arange(T, device=x.device)
        pos_emb = self.position_embedding(pos)   # [T, C]

        x = tok_emb + pos_emb   # [B, T, C]

        return x

In [127]:
embedding = Embedding(
    vocab_size=vocab_size,
    d_model=256,
    block_size=block_size
)

pytorch에서는 클래스 안에 `forward()` 가 있다면 호출할 때

embedding.forward(x) 대신

```
embedding(x)
```

으로 쓸 수 있다.

---

## 4. Causal Self-Attention

<img src="./img/attention.png" width="300">

Casual Self-Attention으ㄹ 이해하기 위해서는 Q, K ,V에 대한 이해가 필요하다.

이는 Query, Key, Value 의 약자로
전체 attention 과정에서의 작동은 Q와 K를 내적 후 softmax 로 만든 확률표에 V를 곱하는 방식이다.

In [ ]:
# x 다시 만들기 (요약)
x, y = next(iter(train_loader))

print(x.shape)
print(y.shape)

x = embedding(x) 

print(x.shape) # [B, T, C] 

torch.Size([32, 8])
torch.Size([32, 8])
torch.Size([32, 8, 256])


| 기호 | 의미 |
|---|---|
| `B` | 배치 사이즈 (`batch size`) |
| `T` | 블록 사이즈 / 시퀀스 길이 (`block size`) |
| `C` | 토큰 벡터의 차원 (`embedding dimension`) |

In [133]:
d_model = 256

query = nn.Linear(d_model, d_model)
key = nn.Linear(d_model, d_model)
value = nn.Linear(d_model, d_model)

Q = query(x) # [B, T, C]
V = value(x) # [B, T, C]
K = key(x) # [B, T, C]

#### Tensor 내적

`[m, n] @ [n, p] = [m. p]`

3차원 tensor의 경우 앞쪽 차원을 배치라고 생각하고 내적하면 된다.

`[B, m, n] @ [B, n, p] = [B, m, p]`

현재 마지막 차원이 

**Q : [T, C]**

**V : [T, C]**

이기에 바로 내적이 불가능하다. 그래서 transpose를 통해 차원을 같게 해줘야 한다.

##### 파이썬 문법

#### torch.tril

`torch.tril(input, diagonal=0)`
- 입력 텐서에서 아래쪽 삼각 부분만 남기고 나머지는 0으로 만드는 함수
```text
1 0 0 0
1 1 0 0
1 1 1 0
1 1 1 1
```



#### tensor.masked_fill

`tensor.masked_fill(mask, value)`

- mask가 True인 위치를 value로 바꾼 새로운 텐서를 반환한다.

In [146]:
x = torch.ones(4, 4)
print(torch.tril(x))

tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])


`tensor.transpose(dim0, dim1)`

dim0과 dim1의 위치를 바꾼다.

```python
x.shape
# [5, 0, 7]

x.transpose(0, 1)
# [0, 5, 7]
```

In [135]:
scores = Q @ K.transpose(-2, -1) #[B, T ,T]
scores = scores /(d_model ** 0.5)

##### 파이썬 문법

#### -$inf$

<img src="./img/softmax.png" width="300">

softmax는 각 벡터 성분에 $e^{성분}$의 평균이다.

그렇기에 mask 미래 토큰을 0으로 만드려면 값을 -inf로 만들면 된다.

In [ ]:
# scores: [B, T, T]

mask = torch.tril(
    torch.ones(T, T, device=x.device)
)

scores = scores.masked_fill(
    mask == 0,
    float("-inf")
)

In [ ]:
attn = torch.softmax(scores, dim=-1)
out = attn @ V  # [B, T, C]

In [ ]:
# 클래스

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, block_size):
        super().__init__()

        self.d_model = d_model

        self.query = nn.Linear(d_model, d_model)
        self.key   = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)

        mask = torch.tril(torch.ones(block_size, block_size))

        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape

        Q = self.query(x)   # [B, T, C]
        K = self.key(x)     # [B, T, C]
        V = self.value(x)   # [B, T, C]

        scores = Q @ K.transpose(-2, -1)

        scores = scores / (d_model ** 0.5)
        # scores = scores / (C ** 0.5)

        scores = scores.masked_fill(
            self.mask[:T, :T] == 0,
            float("-inf")
        )

        attn = torch.softmax(scores, dim=-1)
        # [B, T, T]

        out = attn @ V
        # [B, T, T] @ [B, T, C] = [B, T, C]

        return out

In [142]:
block_size = 8

attention = CausalSelfAttention(
    d_model=256,
    block_size=block_size
)

In [145]:
attention(x)
print(x.shape)

torch.Size([32, 8, 256])


## 5. Multi-Head Attention

<img src="./img/mha.png" width="500">

#### 왜 여러 head를 쓰는가?

single head attention 에서는 하나의 attention만 계산한다.

```
token A는 token B를 얼마나 보는가?
```

Multi Head에서는 여러 관점으로 동시에 계산한다.

| Head | 역할 예시 |
|---|---|
| Head 1 | 문법적 관계 |
| Head 2 | 가까운 단어 관계 |
| Head 3 | 의미적 관계 |
| Head 4 | 다른 패턴 |


실제로 각 head가 반드시 특정 역할을 가진다는 뜻은 아니지만 서로 다른 attention 패턴을 학습할 수 있게 된다.

<img src="./img/mha2.png" width="700">